# Baseline Model Selection

Compare several baseline regressors in the `log1p(target)` setup and select the best model using a competition-aligned validation metric.

In [1]:
import sys

sys.path.append("../")

import json
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
from lightgbm import LGBMRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

from src.loader import Loader

In [3]:
SEED = 42
TEST_SIZE = 0.33
CV = 5

In [4]:
loader = Loader()
df = loader.load(path="../data/processed_data.csv")
df.shape

(4459, 4732)

In [5]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

In [6]:
X_train, X_test, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

In [7]:
models = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "ridge": Ridge(),
    "elasticnet": ElasticNet(max_iter=10000),
    "rand_forest": RandomForestRegressor(random_state=SEED, n_jobs=-1),
    "hgb": HistGradientBoostingRegressor(random_state=SEED),
    "xgb": XGBRegressor(random_state=SEED, n_jobs=-1),
    "lgbm": LGBMRegressor(random_state=SEED, n_jobs=-1, verbosity=-1),
}

linear_models = (Ridge, ElasticNet)

In [8]:
# RMSE in log-space is equivalent to RMSLE for the log-target setup.
results = []

for name, model in models.items():
    if isinstance(model, linear_models):
        model_pipe = Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("model", model),
            ]
        )
    else:
        model_pipe = Pipeline(steps=[("model", model)])

    scores = -cross_val_score(
        estimator=model_pipe,
        X=X_train,
        y=y_train_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )

    results.append(
        {
            "model": name,
            "rmsle_mean": scores.mean(),
            "rmsle_std": scores.std(),
        }
    )

In [9]:
results_df = pd.DataFrame(results).sort_values(by="rmsle_mean")
results_df.style.format({"rmsle_mean": "{:,.3f}", "rmsle_std": "{:,.3f}"}).hide(axis="index")

model,rmsle_mean,rmsle_std
rand_forest,1.439,0.047
lgbm,1.472,0.035
hgb,1.474,0.037
xgb,1.535,0.051
dummy_mean,1.751,0.037
elasticnet,1.751,0.037
ridge,"1,463.442","1,245.177"


In [27]:
best_model_name = results_df.iloc[1]["model"]
best_model = models[best_model_name]
print("Best by CV:", best_model_name)

Best by CV: lgbm


In [28]:
if isinstance(best_model, linear_models):
    best_model_pipe = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", best_model),
        ]
    )
else:
    best_model_pipe = Pipeline(steps=[("model", best_model)])

best_model_pipe.fit(X_train, y_train_log)

y_pred_log = best_model_pipe.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

In [29]:
metrics = pd.DataFrame(
    {
        "metric": ["rmsle", "rmse", "mae", "r2"],
        "value": [
            root_mean_squared_log_error(y_test_raw, y_pred),
            root_mean_squared_error(y_test_raw, y_pred),
            mean_absolute_error(y_test_raw, y_pred),
            r2_score(y_test_raw, y_pred),
        ],
    }
)

metrics.style.format({"value": "{:,.3f}"})

,metric,value
0,rmsle,1.482
1,rmse,"7,326,665.623"
2,mae,"4,216,347.882"
3,r2,0.159


In [30]:
if hasattr(best_model_pipe.named_steps["model"], "feature_importances_"):
    importance_df = pd.DataFrame(
        {
            "feature": X_train.columns,
            "importance": best_model_pipe.named_steps["model"].feature_importances_,
        }
    ).sort_values("importance", ascending=False)
    importance_df.head(20)
elif hasattr(best_model_pipe.named_steps["model"], "coef_"):
    importance_df = pd.DataFrame(
        {
            "feature": X_train.columns,
            "importance": np.abs(best_model_pipe.named_steps["model"].coef_),
        }
    ).sort_values("importance", ascending=False)
    importance_df.head(20)
else:
    print("Feature importance is not available for the selected model.")

In [31]:
ARTIFACTS_DIR = Path("../artifacts/baseline")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "best_model_name": best_model_name,
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "rmsle_test": root_mean_squared_log_error(y_test_raw, y_pred),
    "rmse_test": root_mean_squared_error(y_test_raw, y_pred),
    "mae_test": mean_absolute_error(y_test_raw, y_pred),
    "r2_test": r2_score(y_test_raw, y_pred),
    "model_params": best_model.get_params(),
}

with open(ARTIFACTS_DIR / "baseline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

## Conclusions

- The baseline now evaluates candidate models in the `log1p(target)` setup, which is aligned with the competition metric `RMSLE`.
- Model ranking should be read from cross-validation in log-space, because for this setup it is equivalent to `RMSLE`.
- The selected best model is then refit on the training split, transformed back with `expm1`, and evaluated on the holdout set using `RMSLE` as the primary metric.
- This notebook is now the main baseline entry point; the dedicated log-target notebook is no longer needed as a separate experiment.

## Next experiments

1. Remove near-constant sparse features and compare the result with the current log-target baseline.
2. Tune the best tree-based model for `RMSLE`.
3. Compare feature subsets and target transformations only if they improve the competition metric on validation.